In [0]:
--SELECT version();
--USE DATABASE qdp_database;

CREATE SCHEMA IF NOT EXISTS qdp_appointments;
SET search_path TO qdp_appointments;


-- metadata table
DROP TABLE IF EXISTS metadata;


CREATE TABLE metadata (
	supplier VARCHAR,
	data_product VARCHAR,
	data_product_version VARCHAR,
	schema_name VARCHAR,
	schema_version VARCHAR,
	schema_variant_name VARCHAR,
	schema_variant_version VARCHAR,
	instance VARCHAR,
	instance_version VARCHAR
 );



INSERT INTO metadata
(
  supplier,
  data_product,
  data_product_version,
  schema_name,
  schema_version,
  schema_variant_name,
  schema_variant_version,
  instance,
  instance_version
)
VALUES (
  'quantexa.com',
  'com.quantexa.qdp.appointments',
  '0.1',
  'appointments_all.data_vault',
  '0.1',
  'base',
  '0.1',
  'not yet populated with data',
  ''
);




-- -----------------------------------------------------------
-- create hub_appointment table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS hub_appointment CASCADE;

CREATE TABLE hub_appointment (
    appointment_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
    appointment_entity_id VARCHAR,
  	tennant_id BIGINT NOT NULL DEFAULT 0,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE hub_appointment ADD CONSTRAINT pk_appointment_hubappointment_appointmentid PRIMARY KEY (appointment_id);

COMMENT ON COLUMN hub_appointment.appointment_id IS 'Appointment Identity';




-- -----------------------------------------------------------
-- create sat_appointment table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_appointment;

CREATE TABLE sat_appointment (
    sat_appointment_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
    appointment_id BIGINT NOT NULL,
    load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
    record_source_id BIGINT NOT NULL DEFAULT 1,
	data_source_code VARCHAR,
	data_source_concept_id BIGINT,
	data_source_raw_code VARCHAR,
	data_source_raw_concept_id BIGINT,
	party_id BIGINT,
	encounter_id BIGINT,
	status_code VARCHAR,
	status_concept_id BIGINT,
	status_raw_code VARCHAR,
	status_raw_concept_id BIGINT,
	cancellation_reason_code VARCHAR,
	cancellation_reason_concept_id BIGINT,
	cancellation_reason_raw_code VARCHAR,
	cancellation_reason_raw_concept_id BIGINT,
	class_code VARCHAR,
	class_concept_id BIGINT,
	class_raw_code VARCHAR,
	class_raw_concept_id BIGINT,
	appointment_type_code VARCHAR,
	appointment_type_concept_id BIGINT,
	appointment_type_raw_code VARCHAR,
	appointment_type_raw_concept_id BIGINT,
	priority_code VARCHAR,
	priority_concept_id BIGINT,
	priority_raw_code VARCHAR,
	priority_raw_concept_id BIGINT,
	description VARCHAR,
	start_datetime TIMESTAMP,
	end_datetime TIMESTAMP,
	minutes_duration INT,
	created TIMESTAMP,
	cancellation_date TIMESTAMP,
	patient_instruction TEXT,
	previous_appointment_id BIGINT,
	originating_appointment_id BIGINT,
	recurrence_id INT,
	occurrence_changed BOOLEAN DEFAULT FALSE,
	record_id BIGINT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_appointment ADD CONSTRAINT pk_appointment_satappointment_satappointmentid PRIMARY KEY (sat_appointment_id);
ALTER TABLE sat_appointment ADD CONSTRAINT fk_appointment_satappointment_hubappointment_appointmentid FOREIGN KEY (appointment_id) REFERENCES hub_appointment(appointment_id);

COMMENT ON COLUMN sat_appointment.appointment_id IS 'Appointment Identity';
COMMENT ON COLUMN sat_appointment.sat_appointment_id IS 'Appointment Identity Sat';
COMMENT ON COLUMN sat_appointment.party_id IS 'The Patient Identifier. The patient or group associated with the appointment, if they are to be present (as combust) at the time of the appointment. See https://build.fhir.org/appointment-definitions.html#Appointment.subject';
COMMENT ON COLUMN sat_appointment.encounter_id IS 'The Encounter that this appointment is associated with. Cross-schema reference to Encounter';
COMMENT ON COLUMN sat_appointment.status_code IS 'The overall status of the Appointment. E.g. proposed, pending, booked, arrived, fulfilled, cancelled, noshow, entered-in-error, checked-in, waitlist. See https://hl7.org/fhir/R5/valueset-appointmentstatus.html';
COMMENT ON COLUMN sat_appointment.cancellation_reason_code IS 'The coded reason for the appointment being cancelled. This is often used in reporting and billing. See https://build.fhir.org/appointment-definitions.html#Appointment.cancellationReason';
COMMENT ON COLUMN sat_appointment.class_code IS 'Concepts representing classification of patient encounter. E.g. ambulatory (AMB), emergency (EMER), inpatient (IMP), virtual (VR). See https://build.fhir.org/appointment-definitions.html#Appointment.class';
COMMENT ON COLUMN sat_appointment.appointment_type_code IS 'The style of appointment or patient that has been booked in the slot. E.g. routine, walk-in, check-up, follow-up, emergency. See https://hl7.org/fhir/R5/v2/0276/index.html';
COMMENT ON COLUMN sat_appointment.priority_code IS 'Used to make informed decisions if needing to re-prioritize appointments. E.g. a]routine, urgent, emergency. See https://build.fhir.org/appointment-definitions.html#Appointment.priority';
COMMENT ON COLUMN sat_appointment.description IS 'The brief description of the appointment as would be shown on a subject line in a meeting request, or appointment list. See https://build.fhir.org/appointment-definitions.html#Appointment.description';
COMMENT ON COLUMN sat_appointment.start_datetime IS 'Date/Time that the appointment is to take place. See https://build.fhir.org/appointment-definitions.html#Appointment.start';
COMMENT ON COLUMN sat_appointment.end_datetime IS 'Date/Time that the appointment is to conclude. See https://build.fhir.org/appointment-definitions.html#Appointment.end';
COMMENT ON COLUMN sat_appointment.minutes_duration IS 'Number of minutes that the appointment is to take. This can be less than the duration between the start and end times. See https://build.fhir.org/appointment-definitions.html#Appointment.minutesDuration';
COMMENT ON COLUMN sat_appointment.created IS 'The date that this appointment was initially created. See https://build.fhir.org/appointment-definitions.html#Appointment.created';
COMMENT ON COLUMN sat_appointment.cancellation_date IS 'The date/time describing when the appointment was cancelled. See https://build.fhir.org/appointment-definitions.html#Appointment.cancellationDate';
COMMENT ON COLUMN sat_appointment.patient_instruction IS 'Detailed information and instructions for the patient. E.g. fasting requirements, preparation instructions. See https://build.fhir.org/appointment-definitions.html#Appointment.patientInstruction';
COMMENT ON COLUMN sat_appointment.previous_appointment_id IS 'The previous appointment in a series of related appointments. See https://build.fhir.org/appointment-definitions.html#Appointment.previousAppointment';
COMMENT ON COLUMN sat_appointment.originating_appointment_id IS 'The originating appointment in a recurring set of related appointments. See https://build.fhir.org/appointment-definitions.html#Appointment.originatingAppointment';
COMMENT ON COLUMN sat_appointment.recurrence_id IS 'The sequence number in the recurrence pattern. Used to differentiate individual occurrences within a recurring set. See https://build.fhir.org/appointment-definitions.html#Appointment.recurrenceId';
COMMENT ON COLUMN sat_appointment.occurrence_changed IS 'Indicates whether this occurrence has been modified from the originating appointment recurrence pattern. See https://build.fhir.org/appointment-definitions.html#Appointment.occurrenceChanged';
COMMENT ON COLUMN sat_appointment.record_id IS 'A Unique Identifier for a Data Record within the System that allows Data Ownership, Ownership History, Dynamic Attributes and Metadata to be associated with that Record';



-- -----------------------------------------------------------
-- create sat_appointment_service_category table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_appointment_service_category;

CREATE TABLE sat_appointment_service_category (
	appointment_service_category_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	appointment_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	service_category_code VARCHAR,
	service_category_concept_id BIGINT,
	service_category_raw_code VARCHAR,
	service_category_raw_concept_id BIGINT,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_appointment_service_category ADD CONSTRAINT pk_appointment_satapptsvccat_apptsvcatid PRIMARY KEY (appointment_service_category_id);
ALTER TABLE sat_appointment_service_category ADD CONSTRAINT fk_appointment_satapptsvccat_hubappointment_appointmentid FOREIGN KEY (appointment_id) REFERENCES hub_appointment(appointment_id);

COMMENT ON COLUMN sat_appointment_service_category.appointment_id IS 'Appointment Identity';
COMMENT ON COLUMN sat_appointment_service_category.appointment_service_category_id IS 'Appointment Service Category Identity';
COMMENT ON COLUMN sat_appointment_service_category.service_category_code IS 'A broad categorization of the service that is to be performed during this appointment. E.g. general practice, specialist medical, allied health, dental, mental health, emergency. See https://build.fhir.org/appointment-definitions.html#Appointment.serviceCategory';
COMMENT ON COLUMN sat_appointment_service_category.rank IS 'The recommended order the Appointment Service Category entries should be presented';



-- -----------------------------------------------------------
-- create sat_appointment_service_type table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_appointment_service_type;

CREATE TABLE sat_appointment_service_type (
	appointment_service_type_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	appointment_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	service_type_code VARCHAR,
	service_type_concept_id BIGINT,
	service_type_raw_code VARCHAR,
	service_type_raw_concept_id BIGINT,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_appointment_service_type ADD CONSTRAINT pk_appointment_satapptsvctype_apptsvcypeid PRIMARY KEY (appointment_service_type_id);
ALTER TABLE sat_appointment_service_type ADD CONSTRAINT fk_appointment_satapptsvctype_hubappointment_appointmentid FOREIGN KEY (appointment_id) REFERENCES hub_appointment(appointment_id);

COMMENT ON COLUMN sat_appointment_service_type.appointment_id IS 'Appointment Identity';
COMMENT ON COLUMN sat_appointment_service_type.appointment_service_type_id IS 'Appointment Service Type Identity';
COMMENT ON COLUMN sat_appointment_service_type.service_type_code IS 'The specific service that is to be performed during this appointment. E.g. cardiology, physiotherapy, podiatry, speech pathology, dietary consultation. See https://build.fhir.org/appointment-definitions.html#Appointment.serviceType';
COMMENT ON COLUMN sat_appointment_service_type.rank IS 'The recommended order the Appointment Service Type entries should be presented';



-- -----------------------------------------------------------
-- create sat_appointment_specialty table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_appointment_specialty;

CREATE TABLE sat_appointment_specialty (
	appointment_specialty_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	appointment_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	specialty_code VARCHAR,
	specialty_concept_id BIGINT,
	specialty_raw_code VARCHAR,
	specialty_raw_concept_id BIGINT,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_appointment_specialty ADD CONSTRAINT pk_appointment_satapptspec_apptspecid PRIMARY KEY (appointment_specialty_id);
ALTER TABLE sat_appointment_specialty ADD CONSTRAINT fk_appointment_satapptspec_hubappointment_appointmentid FOREIGN KEY (appointment_id) REFERENCES hub_appointment(appointment_id);

COMMENT ON COLUMN sat_appointment_specialty.appointment_id IS 'Appointment Identity';
COMMENT ON COLUMN sat_appointment_specialty.appointment_specialty_id IS 'Appointment Specialty Identity';
COMMENT ON COLUMN sat_appointment_specialty.specialty_code IS 'The specialty of a practitioner that would be required to perform the service requested in this appointment. E.g. cardiology, dermatology, gastroenterology, general-practice, neurology, orthopaedics, paediatrics, psychiatry. See https://build.fhir.org/appointment-definitions.html#Appointment.specialty';
COMMENT ON COLUMN sat_appointment_specialty.rank IS 'The recommended order the Appointment Specialty entries should be presented';



-- -----------------------------------------------------------
-- create sat_appointment_reason table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_appointment_reason;

CREATE TABLE sat_appointment_reason (
	appointment_reason_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	appointment_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	reason_code VARCHAR,
	reason_concept_id BIGINT,
	reason_raw_code VARCHAR,
	reason_raw_concept_id BIGINT,
	reason_reference_type VARCHAR,
	reason_reference_id BIGINT,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_appointment_reason ADD CONSTRAINT pk_appointment_satapptreason_apptreasonid PRIMARY KEY (appointment_reason_id);
ALTER TABLE sat_appointment_reason ADD CONSTRAINT fk_appointment_satapptreason_hubappointment_appointmentid FOREIGN KEY (appointment_id) REFERENCES hub_appointment(appointment_id);

COMMENT ON COLUMN sat_appointment_reason.appointment_id IS 'Appointment Identity';
COMMENT ON COLUMN sat_appointment_reason.appointment_reason_id IS 'Appointment Reason Identity';
COMMENT ON COLUMN sat_appointment_reason.reason_code IS 'The coded reason that this appointment is being scheduled. SNOMED CT Clinical Finding coded. See https://build.fhir.org/appointment-definitions.html#Appointment.reason';
COMMENT ON COLUMN sat_appointment_reason.reason_reference_type IS 'Type of the reason reference resource (e.g. Condition, Procedure, Observation, ImmunizationRecommendation)';
COMMENT ON COLUMN sat_appointment_reason.reason_reference_id IS 'Identifier of the reason reference resource';
COMMENT ON COLUMN sat_appointment_reason.rank IS 'The recommended order the Appointment Reason entries should be presented';



-- -----------------------------------------------------------
-- create sat_appointment_participant table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_appointment_participant;

CREATE TABLE sat_appointment_participant (
	appointment_participant_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	appointment_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	participant_type_code VARCHAR,
	participant_type_concept_id BIGINT,
	participant_type_raw_code VARCHAR,
	participant_type_raw_concept_id BIGINT,
	actor_party_id BIGINT,
	actor_location_id BIGINT,
	actor_device_id BIGINT,
	actor_healthcare_service_code VARCHAR,
	actor_healthcare_service_concept_id BIGINT,
	actor_healthcare_service_raw_code VARCHAR,
	actor_healthcare_service_raw_concept_id BIGINT,
	required_code VARCHAR,
	required_concept_id BIGINT,
	required_raw_code VARCHAR,
	required_raw_concept_id BIGINT,
	participation_status_code VARCHAR,
	participation_status_concept_id BIGINT,
	participation_status_raw_code VARCHAR,
	participation_status_raw_concept_id BIGINT,
	participation_period_start TIMESTAMP,
	participation_period_end TIMESTAMP,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_appointment_participant ADD CONSTRAINT pk_appointment_satapptpart_apptpartid PRIMARY KEY (appointment_participant_id);
ALTER TABLE sat_appointment_participant ADD CONSTRAINT fk_appointment_satapptpart_hubappointment_appointmentid FOREIGN KEY (appointment_id) REFERENCES hub_appointment(appointment_id);

COMMENT ON COLUMN sat_appointment_participant.appointment_id IS 'Appointment Identity';
COMMENT ON COLUMN sat_appointment_participant.appointment_participant_id IS 'Appointment Participant Identity';
COMMENT ON COLUMN sat_appointment_participant.participant_type_code IS 'Role of participant in the appointment. E.g. ADM (admitter), ATND (attender), CALLBCK (callback contact), CON (consultant), DIS (discharger), ESC (escort), REF (referrer), SPRF (secondary performer), PPRF (primary performer), PART (participation). See https://hl7.org/fhir/R5/valueset-encounter-participant-type.html';
COMMENT ON COLUMN sat_appointment_participant.actor_party_id IS 'The Person, Organization, or CareTeam participating in the appointment. See https://build.fhir.org/appointment-definitions.html#Appointment.participant.actor';
COMMENT ON COLUMN sat_appointment_participant.actor_location_id IS 'The Location participating in the appointment (e.g. room, ward). Cross-schema reference to Location';
COMMENT ON COLUMN sat_appointment_participant.actor_device_id IS 'The Device participating in the appointment. Cross-schema reference to Device';
COMMENT ON COLUMN sat_appointment_participant.actor_healthcare_service_code IS 'The HealthcareService participating in the appointment';
COMMENT ON COLUMN sat_appointment_participant.required_code IS 'Whether this participant is required to be present at the meeting. E.g. required, optional, information-only. See https://hl7.org/fhir/R5/valueset-participantrequired.html';
COMMENT ON COLUMN sat_appointment_participant.participation_status_code IS 'Participation status of the actor. E.g. accepted, declined, tentative, needs-action. See https://hl7.org/fhir/R5/valueset-participationstatus.html';
COMMENT ON COLUMN sat_appointment_participant.participation_period_start IS 'Start of the participation period of the actor. See https://build.fhir.org/appointment-definitions.html#Appointment.participant.period';
COMMENT ON COLUMN sat_appointment_participant.participation_period_end IS 'End of the participation period of the actor';
COMMENT ON COLUMN sat_appointment_participant.rank IS 'The recommended order the Appointment Participant entries should be presented';



-- -----------------------------------------------------------
-- create sat_appointment_requested_period table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_appointment_requested_period;

CREATE TABLE sat_appointment_requested_period (
	appointment_requested_period_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	appointment_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	requested_period_start TIMESTAMP,
	requested_period_end TIMESTAMP,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_appointment_requested_period ADD CONSTRAINT pk_appointment_satapptreqperiod_apptreqperiodid PRIMARY KEY (appointment_requested_period_id);
ALTER TABLE sat_appointment_requested_period ADD CONSTRAINT fk_appointment_satapptreqperiod_hubappointment_appointmentid FOREIGN KEY (appointment_id) REFERENCES hub_appointment(appointment_id);

COMMENT ON COLUMN sat_appointment_requested_period.appointment_id IS 'Appointment Identity';
COMMENT ON COLUMN sat_appointment_requested_period.appointment_requested_period_id IS 'Appointment Requested Period Identity';
COMMENT ON COLUMN sat_appointment_requested_period.requested_period_start IS 'Start of the potential date/time interval requested to allocate the appointment within. See https://build.fhir.org/appointment-definitions.html#Appointment.requestedPeriod';
COMMENT ON COLUMN sat_appointment_requested_period.requested_period_end IS 'End of the potential date/time interval requested to allocate the appointment within';
COMMENT ON COLUMN sat_appointment_requested_period.rank IS 'The recommended order the Appointment Requested Period entries should be presented';



-- -----------------------------------------------------------
-- create sat_appointment_notes table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_appointment_notes;

CREATE TABLE sat_appointment_notes (
	appointment_note_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	appointment_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	author_reference_party_id BIGINT,
	author_string VARCHAR,
	time TIMESTAMP,
	text TEXT NOT NULL,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_appointment_notes ADD CONSTRAINT pk_appointment_satapptnotes_apptnoteid PRIMARY KEY (appointment_note_id);
ALTER TABLE sat_appointment_notes ADD CONSTRAINT fk_appointment_satapptnotes_hubappointment_appointmentid FOREIGN KEY (appointment_id) REFERENCES hub_appointment(appointment_id);

COMMENT ON COLUMN sat_appointment_notes.appointment_id IS 'Appointment Identity';
COMMENT ON COLUMN sat_appointment_notes.appointment_note_id IS 'Appointment Note Identity';
COMMENT ON COLUMN sat_appointment_notes.author_reference_party_id IS 'The clinical Party Identifier that recorded this annotation. E.g. Practitioner, PractitionerRole, Patient, RelatedPerson, Organization. See https://build.fhir.org/appointment-definitions.html#Appointment.note';
COMMENT ON COLUMN sat_appointment_notes.author_string IS 'The Individual that recorded this annotation (as free text)';
COMMENT ON COLUMN sat_appointment_notes.time IS 'When this annotation was recorded';
COMMENT ON COLUMN sat_appointment_notes.text IS 'Additional comments about the appointment. See https://build.fhir.org/appointment-definitions.html#Appointment.note';




-- -----------------------------------------------------------
-- create view_appointments view
-- -----------------------------------------------------------
CREATE OR REPLACE VIEW view_appointments AS
SELECT
  h.tennant_id,
  t.tennant_name,
  h.appointment_id,
	h.appointment_entity_id,
	h.period_start AS hub_period_start,
	h.period_end AS hub_period_end,
  s.sat_appointment_id,
  s.party_id,
  s.encounter_id,
  s.status_code,
  s.status_concept_id,
  s.status_raw_code,
  s.status_raw_concept_id,
  s.cancellation_reason_code,
  s.cancellation_reason_concept_id,
  s.cancellation_reason_raw_code,
  s.cancellation_reason_raw_concept_id,
  s.class_code,
  s.class_concept_id,
  s.class_raw_code,
  s.class_raw_concept_id,
  s.appointment_type_code,
  s.appointment_type_concept_id,
  s.appointment_type_raw_code,
  s.appointment_type_raw_concept_id,
  s.priority_code,
  s.priority_concept_id,
  s.priority_raw_code,
  s.priority_raw_concept_id,
  s.description,
  s.start_datetime,
  s.end_datetime,
  s.minutes_duration,
  s.created,
  s.cancellation_date,
  s.patient_instruction,
  s.previous_appointment_id,
  s.originating_appointment_id,
  s.recurrence_id,
  s.occurrence_changed,
  s.record_id,
	s.period_start,
	s.period_end,
	s.load_datetime,
	s.record_source_id,
	s.data_source_code,
	s.data_source_concept_id,
	s.data_source_raw_code,
	s.data_source_raw_concept_id

    FROM budd.qdp_appointments.hub_appointment h
    JOIN budd.qdp_appointments.sat_appointment s
      ON h.appointment_id = s.appointment_id
    JOIN budd.qdp_refcodes.tennant t
      on h.tennant_id = t.tennant_id
;

SELECT * from view_appointments;


-- -----------------------------------------------------------
-- create view_appointment_participants view
-- -----------------------------------------------------------
CREATE OR REPLACE VIEW view_appointment_participants AS
SELECT
  h.tennant_id,
  t.tennant_name,
  h.appointment_id,
	h.appointment_entity_id,
  s.status_code,
  s.appointment_type_code,
  s.description,
  s.start_datetime,
  s.end_datetime,
  s.party_id,
	ap.appointment_participant_id,
	ap.participant_type_code,
	ap.participant_type_concept_id,
	ap.participant_type_raw_code,
	ap.participant_type_raw_concept_id,
	ap.actor_party_id,
	ap.actor_location_id,
	ap.actor_device_id,
	ap.actor_healthcare_service_code,
	ap.actor_healthcare_service_concept_id,
	ap.required_code,
	ap.required_concept_id,
	ap.participation_status_code,
	ap.participation_status_concept_id,
	ap.participation_period_start,
	ap.participation_period_end,
	ap.rank AS participant_rank,
	ap.period_start AS participant_period_start,
	ap.period_end AS participant_period_end,
	h.period_start AS hub_period_start,
	h.period_end AS hub_period_end,
	s.period_start,
	s.period_end,
	s.load_datetime,
	s.record_source_id,
	s.data_source_code,
	s.data_source_concept_id,
	s.data_source_raw_code,
	s.data_source_raw_concept_id

    FROM budd.qdp_appointments.hub_appointment h
    JOIN budd.qdp_appointments.sat_appointment s
      ON h.appointment_id = s.appointment_id
    JOIN budd.qdp_appointments.sat_appointment_participant ap
      ON h.appointment_id = ap.appointment_id
    JOIN budd.qdp_refcodes.tennant t
      on h.tennant_id = t.tennant_id
;

SELECT * from view_appointment_participants;


-- -----------------------------------------------------------
-- create view_appointment_reasons view
-- -----------------------------------------------------------
CREATE OR REPLACE VIEW view_appointment_reasons AS
SELECT
  h.tennant_id,
  t.tennant_name,
  h.appointment_id,
	h.appointment_entity_id,
  s.status_code,
  s.appointment_type_code,
  s.description,
  s.start_datetime,
  s.end_datetime,
  s.party_id,
	ar.appointment_reason_id,
	ar.reason_code,
	ar.reason_concept_id,
	ar.reason_raw_code,
	ar.reason_raw_concept_id,
	ar.reason_reference_type,
	ar.reason_reference_id,
	ar.rank AS reason_rank,
	ar.period_start AS reason_period_start,
	ar.period_end AS reason_period_end,
	h.period_start AS hub_period_start,
	h.period_end AS hub_period_end,
	s.period_start,
	s.period_end,
	s.load_datetime,
	s.record_source_id,
	s.data_source_code,
	s.data_source_concept_id,
	s.data_source_raw_code,
	s.data_source_raw_concept_id

    FROM budd.qdp_appointments.hub_appointment h
    JOIN budd.qdp_appointments.sat_appointment s
      ON h.appointment_id = s.appointment_id
    JOIN budd.qdp_appointments.sat_appointment_reason ar
      ON h.appointment_id = ar.appointment_id
    JOIN budd.qdp_refcodes.tennant t
      on h.tennant_id = t.tennant_id
;

SELECT * from view_appointment_reasons;


-- -----------------------------------------------------------
-- create view_appointment_services view
-- -----------------------------------------------------------
CREATE OR REPLACE VIEW view_appointment_services AS
SELECT
  h.tennant_id,
  t.tennant_name,
  h.appointment_id,
	h.appointment_entity_id,
  s.status_code,
  s.appointment_type_code,
  s.description,
  s.start_datetime,
  s.end_datetime,
  s.party_id,
	asc2.appointment_service_category_id,
	asc2.service_category_code,
	asc2.service_category_concept_id,
	asc2.service_category_raw_code,
	asc2.service_category_raw_concept_id,
	ast.appointment_service_type_id,
	ast.service_type_code,
	ast.service_type_concept_id,
	ast.service_type_raw_code,
	ast.service_type_raw_concept_id,
	h.period_start AS hub_period_start,
	h.period_end AS hub_period_end,
	s.period_start,
	s.period_end,
	s.load_datetime,
	s.record_source_id,
	s.data_source_code,
	s.data_source_concept_id,
	s.data_source_raw_code,
	s.data_source_raw_concept_id

    FROM budd.qdp_appointments.hub_appointment h
    JOIN budd.qdp_appointments.sat_appointment s
      ON h.appointment_id = s.appointment_id
    LEFT JOIN budd.qdp_appointments.sat_appointment_service_category asc2
      ON h.appointment_id = asc2.appointment_id
    LEFT JOIN budd.qdp_appointments.sat_appointment_service_type ast
      ON h.appointment_id = ast.appointment_id
    JOIN budd.qdp_refcodes.tennant t
      on h.tennant_id = t.tennant_id
;

SELECT * from view_appointment_services;


/**

**/
